In [7]:
#! pip install kafka-python

In [8]:
from kafka import KafkaAdminClient

In [9]:
# create connection to kafka broker 

admin= KafkaAdminClient(
    bootstrap_servers="localhost:9092"
)
print("connected successfully")

connected successfully


In [10]:
#list all topics 
topics = admin.list_topics()
print(topics)

[]


In [11]:
admin.describe_cluster()

{'throttle_time_ms': 0,
 'brokers': [{'node_id': 1, 'host': 'localhost', 'port': 9092, 'rack': None}],
 'cluster_id': '5L6g3nShT-eMCtK--X86sw',
 'controller_id': 1,
 'authorized_operations': ['CREATE',
  'ALTER',
  'DESCRIBE',
  'CLUSTER_ACTION',
  'DESCRIBE_CONFIGS',
  'ALTER_CONFIGS',
  'IDEMPOTENT_WRITE']}

In [12]:
from kafka.admin import NewTopic

In [13]:
# define topic configuration 
sensor_topic = NewTopic(
    name= "sensor-telemetry",
    num_partitions=3,
    replication_factor=1
)

In [14]:
# create  topic in kafka 
admin.create_topics(
    new_topics=[sensor_topic],
    validate_only=False
)

print("topics created sucessfully")

topics created sucessfully


In [15]:
# verify topic exists  or not
topics =admin.list_topics()
print(topics)

['sensor-telemetry']


In [16]:
#inspect topic metadata
from kafka import KafkaConsumer
consumer = KafkaConsumer(
    bootstrap_servers= "localhost:9092"
)

partitions= consumer.partitions_for_topic(
    "sensor-telemetry"
)
print(partitions)

{0, 1, 2}


In [17]:
# import kafka
from kafka import KafkaProducer
import json

In [18]:
#create producer connection
producer= KafkaProducer(
bootstrap_servers="localhost:9092",
value_serializer= lambda v: json.dumps(v).encode("utf-8")
)

In [19]:
# create sensor event 

sensor_event= {
    "sensor_id": "S101",
    "temprature": 24.5,
    "humidity": 60
}

print(sensor_event)

{'sensor_id': 'S101', 'temprature': 24.5, 'humidity': 60}


In [20]:
# send message to kafka 
future = producer.send(
    "sensor-telemetry",
    value= sensor_event
)

print("message sent")

message sent


In [21]:
sensor_event= [
{"sensor_id": "S101","temprature": 24},
{"sensor_id": "S102","temprature": 26},
{"sensor_id": "S103","temprature": 45},
{"sensor_id": "S104","temprature": 70},
{"sensor_id": "S105","temprature": 24},
{"sensor_id": "S106","temprature": 26},
{"sensor_id": "S107","temprature": 45},
{"sensor_id": "S108","temprature": 70}   
]

for event in sensor_event:
    metadata=producer.send(
        "sensor-telemetry",
        key= event["sensor_id"].encode("utf-8"),
        value=event
    ).get()
    print(
        f"partition={metadata}",
        f"offset={metadata.offset}"
    )

partition=RecordMetadata(topic='sensor-telemetry', partition=0, topic_partition=TopicPartition(topic='sensor-telemetry', partition=0), offset=0, timestamp=1781970465064, checksum=None, serialized_key_size=4, serialized_value_size=39, serialized_header_size=-1) offset=0
partition=RecordMetadata(topic='sensor-telemetry', partition=1, topic_partition=TopicPartition(topic='sensor-telemetry', partition=1), offset=1, timestamp=1781970465071, checksum=None, serialized_key_size=4, serialized_value_size=39, serialized_header_size=-1) offset=1
partition=RecordMetadata(topic='sensor-telemetry', partition=0, topic_partition=TopicPartition(topic='sensor-telemetry', partition=0), offset=1, timestamp=1781970465076, checksum=None, serialized_key_size=4, serialized_value_size=39, serialized_header_size=-1) offset=1
partition=RecordMetadata(topic='sensor-telemetry', partition=0, topic_partition=TopicPartition(topic='sensor-telemetry', partition=0), offset=2, timestamp=1781970465085, checksum=None, seria

In [22]:
producer.flush()
print("all message sent")

all message sent


In [26]:
#create our first consumer
from kafka import KafkaConsumer

In [34]:
import json
consumer= KafkaConsumer(
"sensor-telemetry",
bootstrap_servers="localhost:9092",
auto_offset_reset="earliest",
value_deserializer= lambda m:json.loads(m.decode("utf-8")),
consumer_timeout_ms=500
)
print(consumer)

In [35]:
#read message
for msg in consumer:
    print(msg)

ConsumerRecord(topic='sensor-telemetry', partition=0, leader_epoch=0, offset=0, timestamp=1781970465064, timestamp_type=0, key=b'S101', value={'sensor_id': 'S101', 'temprature': 24}, headers=[], checksum=None, serialized_key_size=4, serialized_value_size=39, serialized_header_size=-1)
ConsumerRecord(topic='sensor-telemetry', partition=0, leader_epoch=0, offset=1, timestamp=1781970465076, timestamp_type=0, key=b'S103', value={'sensor_id': 'S103', 'temprature': 45}, headers=[], checksum=None, serialized_key_size=4, serialized_value_size=39, serialized_header_size=-1)
ConsumerRecord(topic='sensor-telemetry', partition=0, leader_epoch=0, offset=2, timestamp=1781970465085, timestamp_type=0, key=b'S104', value={'sensor_id': 'S104', 'temprature': 70}, headers=[], checksum=None, serialized_key_size=4, serialized_value_size=39, serialized_header_size=-1)
ConsumerRecord(topic='sensor-telemetry', partition=0, leader_epoch=0, offset=3, timestamp=1781970465094, timestamp_type=0, key=b'S105', value=

In [39]:
for msg in consumer:
    print(
        f""" 
topic : {msg.topic}
partition : {msg.partition}
offset : {msg.offset}
value : {msg.value}
"""
 )

In [40]:
#check consumer position
for partition in consumer.assignment():
    position = consumer.position(partition)
    print(f"{partition} -> next offset = {position}")

TopicPartition(topic='sensor-telemetry', partition=0) -> next offset = 4
TopicPartition(topic='sensor-telemetry', partition=1) -> next offset = 3
TopicPartition(topic='sensor-telemetry', partition=2) -> next offset = 2
